In [18]:
import pickle
import numpy as np

def predict_pv_status(voltage_input, power_input):
    """
    Takes live measurements from the TCT PV Array, processes them, 
    and predicts both the optimal configuration and the current shading state.
    """
    try:
        with open('config_xgb_model_eng.pkl', 'rb') as f:
            config_model = pickle.load(f)
        with open('shading_xgb_model_eng.pkl', 'rb') as f:
            shading_model = pickle.load(f)
        with open('label_encoder.pkl', 'rb') as f:
            label_encoder = pickle.load(f)
    except FileNotFoundError:
        return "Error: Model files not found. Make sure training is complete!"

    # Feature Extraction Pipeline
    epsilon = 1e-5
    
    est_current = power_input / (voltage_input + epsilon)
    static_resistance = (voltage_input**2) / (power_input + epsilon)
    power_v2_ratio = power_input / (voltage_input**2 + epsilon)
    v_midpoint_deviation = voltage_input - 60.0
    current_deficit = max(0.0, 8.0 - est_current)
    v_p_interaction = voltage_input * power_input
    normalized_power = power_input / 270.0
    
    feature_vector = np.array([[
        voltage_input, power_input, est_current, static_resistance,
        power_v2_ratio, v_midpoint_deviation, current_deficit,
        v_p_interaction, normalized_power
    ]])

    encoded_config = config_model.predict(feature_vector)
    optimal_config = label_encoder.inverse_transform(encoded_config)[0]
    
    confidence = np.max(config_model.predict_proba(feature_vector)) * 100
    
    predicted_shading_flat = shading_model.predict(feature_vector)[0]
    shading_matrix_3x3 = predicted_shading_flat.reshape(3, 3)
    
    shading_matrix_3x3 = np.clip(shading_matrix_3x3, 100.0, 1000.0)

    print(f" Array  Input : {voltage_input:.2f} V  , {power_input:.2f} W")
    print(f" RECOMMENDED SWITCH LAYOUT : Configuration Label {optimal_config}")
    print(f" Optimization Confidence   : {confidence:.2f}%")
    print(" PREDICTED 3x3 MODULE SOLAR IRRADIANCE (W/m²):")
    
    # Nice structured print out of the rows
    for row in shading_matrix_3x3:
        print(f"   [ {row[0]:6.1f}  {row[1]:6.1f}  {row[2]:6.1f} ]")
    
    return optimal_config, shading_matrix_3x3

if __name__ == "__main__":
    
    measured_v1 = 101
    measured_p1 = 350
    predict_pv_status(measured_v1, measured_p1)
    
   

 Array  Input : 101.00 V  , 350.00 W
 RECOMMENDED SWITCH LAYOUT : Configuration Label 4
 Optimization Confidence   : 83.84%
 PREDICTED 3x3 MODULE SOLAR IRRADIANCE (W/m²):
   [  237.6   652.3   392.0 ]
   [  781.3   615.0   299.3 ]
   [  225.0   693.2   226.8 ]
